# Movie Plot Compressor

This notebook extracts movie plots from Wikipedia and summarizes them using BART model.

## Step 1: Install Required Packages

In [1]:
!pip install transformers
!pip install torch
!pip install wikipedia
!pip install beautifulsoup4
!pip install requests
!pip install lxml

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


## Step 2: Import All Libraries

In [2]:
from transformers import BartForConditionalGeneration, BartTokenizer
import wikipedia
import requests
from bs4 import BeautifulSoup
import re
import warnings
warnings.filterwarnings('ignore')

print("libraries imported successfully!")

/Users/nandu/Library/Python/3.11/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/nandu/Library/Python/3.11/lib/python/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/nand

libraries imported successfully!


## Step 3: Load BART Model (This may take 5-10 minutes on first run)

In [3]:
print("Loading BART model... This may take a few minutes on first run.")
print("Model size: ~1.6GB - Please be patient!\n")

model_name = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

print("\n Model loaded successfully!")
print(" Tokenizer loaded successfully!")

Loading BART model... This may take a few minutes on first run.
Model size: ~1.6GB - Please be patient!



Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 511/511 [00:03<00:00, 133.82it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]   



 Model loaded successfully!
 Tokenizer loaded successfully!


## Step 4: Define All Helper Functions

In [5]:
def extract_plot_section(content):
    """
    Extract the plot section from Wikipedia content
    """

    plot_match = re.search(r'== Plot ==\s*(.*?)(?=\n==|$)', content, re.DOTALL | re.IGNORECASE)
    
    if plot_match:
        plot = plot_match.group(1).strip()
        plot = re.sub(r'\[\d+\]', '', plot)
    synopsis_match = re.search(r'== Synopsis ==\s*(.*?)(?=\n==|$)', content, re.DOTALL | re.IGNORECASE)
    if synopsis_match:
        synopsis = synopsis_match.group(1).strip()
        synopsis = re.sub(r'\[\d+\]', '', synopsis)
        return synopsis
    return content[:2000]

def scrape_movie_metadata(url):
    """
    Scrape genre and year from Wikipedia page
    """
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')
        infobox = soup.find('table', {'class': 'infobox'})
        
        genre = "Unknown"
        year = "Unknown"
        
        if infobox:
            rows = infobox.find_all('tr')
            
            for row in rows:
                header = row.find('th')
                if header:
                    header_text = header.get_text().strip()
                    if 'Genre' in header_text:
                        data = row.find('td')
                        if data:
                            genre_text = data.get_text(separator=' ', strip=True)
                            genre = re.sub(r'\[\d+\]', '', genre_text).strip()
                            if len(genre) > 100:
                                genre = genre[:100] + "..."
                    if 'Release' in header_text or 'Released' in header_text:
                        data = row.find('td')
                        if data:
                            date_text = data.get_text()
                            year_match = re.search(r'\b(19|20)\d{2}\b', date_text)
                            if year_match:
                                year = year_match.group(0)
        
        return genre, year
    
    except Exception as e:
        print(f"Warning: Could not extract metadata: {e}")
        return "Unknown", "Unknown"

def get_movie_info_from_wikipedia(movie_name):
    """
    Extract movie information from Wikipedia
    Returns: plot, genre, year, title
    """
    try:
        search_results = wikipedia.search(movie_name + " film", results=5)
        
        if not search_results:
            search_results = wikipedia.search(movie_name, results=5)
            if not search_results:
                return None, None, None, None
        page_title = search_results[0]
        page = wikipedia.page(page_title, auto_suggest=False)
        content = page.content
        plot = extract_plot_section(content)
        url = page.url
        genre, year = scrape_movie_metadata(url)
        return plot, genre, year, page_title
    
    except wikipedia.exceptions.DisambiguationError as e:
        try:
            page = wikipedia.page(e.options[0], auto_suggest=False)
            content = page.content
            plot = extract_plot_section(content)
            url = page.url
            genre, year = scrape_movie_metadata(url)
            return plot, genre, year, e.options[0]
        except:
            return None, None, None, None
    
    except Exception as e:
        print(f"Error fetching Wikipedia data: {str(e)}")
        return None, None, None, None

def summarize_plot(plot, max_length=200, min_length=100):
    """
    Summarize the plot using BART model
    Uses the global model and tokenizer variables
    """
    try:
        if 'model' not in globals() or 'tokenizer' not in globals():
            return "Error: Model not loaded. Please run the model loading cell first."
        inputs = tokenizer([plot], max_length=1024, return_tensors='pt', truncation=True)
        summary_ids = model.generate(
            inputs['input_ids'], 
            max_length=max_length, 
            min_length=min_length, 
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True
        )
        summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        return summary
    except Exception as e:
        return f"Error during summarization: {str(e)}"

def format_summary_as_points(summary):
    """
    Format summary into bullet points
    """
    sentences = re.split(r'(?<=[.!?])\s+', summary)
    return [s.strip() for s in sentences if s.strip()]

print("All functions defined successfully!")

All functions defined successfully!


## Step 5: Define Main Summary Function

In [6]:
def get_movie_summary(movie_name):
    """
    Main function to fetch and display movie summary
    """
    print(f"\n{'='*70}")
    print(f"Searching for: {movie_name}")
    print(f"{'='*70}\n")
    plot, genre, year, title = get_movie_info_from_wikipedia(movie_name)
    
    if plot is None:
        print("Movie not found on Wikipedia. Please check the spelling or try another title.")
        return
    
    print(f"Found: {title}\n")
    print(f"📽️  MOVIE INFORMATION")
    print(f"{'─'*70}")
    print(f"Title:  {title}")
    print(f"Genre:  {genre}")
    print(f"Year:   {year}")
    print(f"{'─'*70}\n")
    print(f"Plot length: {len(plot)} characters\n")
    print("Generating summary (this may take 10-30 seconds)...\n")
    summary = summarize_plot(plot)
    if summary.startswith("Error"):
        print(f"{summary}")
        return
    print(f"SUMMARY")
    print(f"{'─'*70}")
    print(summary)
    print(f"{'─'*70}\n")
    sentences = format_summary_as_points(summary)
    print(f" KEY POINTS (10-Line Format)")
    print(f"{'─'*70}")
    for i, sentence in enumerate(sentences[:10], 1):
        print(f"{i}. {sentence}")
    print(f"{'─'*70}\n")
    
    print(f"{'='*70}\n")

print("Main function ready!")

Main function ready!


---

# 🎬 NOW USE THIS CELL TO GET MOVIE SUMMARIES!

**Make sure you've run ALL the cells above first!**

Simply change the movie name and run this cell:

In [7]:
movie_name = "Inception"

get_movie_summary(movie_name)


Searching for: Inception

Found: Inception

📽️  MOVIE INFORMATION
──────────────────────────────────────────────────────────────────────
Title:  Inception
Genre:  Unknown
Year:   2010
──────────────────────────────────────────────────────────────────────

Plot length: 2000 characters

Generating summary (this may take 10-30 seconds)...

SUMMARY
──────────────────────────────────────────────────────────────────────
Inception is a 2010 science fiction action film written and directed by Christopher Nolan. The film stars Leonardo DiCaprio as a professional thief who steals information by infiltrating the subconscious of his targets. The ensemble cast includes Ken Watanabe, Joseph Gordon-Levitt, Marion Cotillard, Elliot Page, Tom Hardy, Cillian Murphy, Tom Berenger, Dileep Rao, and Michael Caine. Inception grossed $839 million worldwide, becoming the fourth-highest-grossing film of 2010.
──────────────────────────────────────────────────────────────────────

 KEY POINTS (10-Line Format)
─

---

## Try Different Movies

In [8]:
get_movie_summary("The Dark Knight")


Searching for: The Dark Knight

Found: The Dark Knight

📽️  MOVIE INFORMATION
──────────────────────────────────────────────────────────────────────
Title:  The Dark Knight
Genre:  Unknown
Year:   2008
──────────────────────────────────────────────────────────────────────

Plot length: 2000 characters

Generating summary (this may take 10-30 seconds)...

SUMMARY
──────────────────────────────────────────────────────────────────────
The Dark Knight is a 2008 superhero film directed by Christopher Nolan. Based on the DC Comics superhero Batman, it is the sequel to Batman Begins (2005) and the second installment in The Dark Knight trilogy. The ensemble cast includes Christian Bale, Michael Caine, Heath Ledger, Gary Oldman, Aaron Eckhart, Maggie Gyllenhaal, and Morgan Freeman. The film was marketed with an innovative interactive viral campaign that initially focused on countering criticism of Ledger's casting by those who believed he was a poor choice.
────────────────────────────────────

In [10]:

get_movie_summary("Interstellar")


Searching for: Interstellar

Found: Interstellar (film)

📽️  MOVIE INFORMATION
──────────────────────────────────────────────────────────────────────
Title:  Interstellar (film)
Genre:  Unknown
Year:   2014
──────────────────────────────────────────────────────────────────────

Plot length: 2000 characters

Generating summary (this may take 10-30 seconds)...

SUMMARY
──────────────────────────────────────────────────────────────────────
Interstellar is a 2014 epic science fiction film directed by Christopher Nolan. It features an ensemble cast led by Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, and Michael Caine. The film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind. It was a commercial success, grossing $681 million worldwide during its initial theatrical run, and $773.8 million with subsequent releases. Interstellar was nominated for five awards at the 87th Academy Awards.
────────────

In [11]:

get_movie_summary("The Matrix")


Searching for: The Matrix

Found: The Matrix

📽️  MOVIE INFORMATION
──────────────────────────────────────────────────────────────────────
Title:  The Matrix
Genre:  Unknown
Year:   1999
──────────────────────────────────────────────────────────────────────

Plot length: 2000 characters

Generating summary (this may take 10-30 seconds)...

SUMMARY
──────────────────────────────────────────────────────────────────────
The Matrix is a 1999 science fiction action film written and directed by the Wachowskis. The first installment in the Matrix film series, it stars Keanu Reeves, Laurence Fishburne, Carrie-Anne Moss, Hugo Weaving, and Joe Pantoliano. It depicts a dystopian future in which humanity is unknowingly trapped inside the Matrix. The film was a box office success, grossing over $460 million on a $63 million budget. It became the highest-grossing Warner Bros. film of 1999.
──────────────────────────────────────────────────────────────────────

 KEY POINTS (10-Line Format)
─────────

---

## Summarize Multiple Movies at Once

In [9]:
movies = [
    "The Shawshank Redemption",
    "The Godfather",
    "Pulp Fiction"
]

for movie in movies:
    get_movie_summary(movie)
    print("\n" + "="*70 + "\n")


Searching for: The Shawshank Redemption

Found: The Shawshank Redemption

📽️  MOVIE INFORMATION
──────────────────────────────────────────────────────────────────────
Title:  The Shawshank Redemption
Genre:  Unknown
Year:   1994
──────────────────────────────────────────────────────────────────────

Plot length: 2000 characters

Generating summary (this may take 10-30 seconds)...

SUMMARY
──────────────────────────────────────────────────────────────────────
The Shawshank Redemption is a 1994 American drama film written and directed by Frank Darabont. It is based on the 1982 Stephen King novella Rita Hayworth and Shawshanks Redemption. The film tells the story of banker Andy Dufresne (Tim Robbins) who is sentenced to life in prison for the murders of his wife and her lover. Over the following two decades, he befriends a fellow prisoner, contraband smuggler Ellis "Red" Redding (Morgan Freeman), and becomes instrumental in a money laundering operation led by the prison warden.
─────────

---

## Interactive Input (Keep Entering Movies)

In [12]:
while True:
    movie_name = input("\nEnter movie name (or 'quit' to exit): ")
    
    if movie_name.lower() in ['quit', 'exit', 'q']:
        print("\nGoodbye!")
        break
    
    if movie_name.strip():
        get_movie_summary(movie_name)
    else:
        print("Please enter a valid movie name.")


Searching for: munnariyippu

Found: Munnariyippu

📽️  MOVIE INFORMATION
──────────────────────────────────────────────────────────────────────
Title:  Munnariyippu
Genre:  Unknown
Year:   2014
──────────────────────────────────────────────────────────────────────

Plot length: 2000 characters

Generating summary (this may take 10-30 seconds)...

SUMMARY
──────────────────────────────────────────────────────────────────────
Munnariyippu (transl. Warning) is a 2014 Indian Malayalam-language psychological experimental thriller film directed by Venu and produced by Ranjith. The film stars Mammootty and Aparna Gopinath, with Nedumudi Venu, Joy Mathew, Prathap Pothan, Sreeraman, Renji Panicker, Saiju Kurup, Joshy Mathew and Sudheesh in supporting roles. It released on 22 August 2014 to critical and commercial acclaim.
──────────────────────────────────────────────────────────────────────

 KEY POINTS (10-Line Format)
──────────────────────────────────────────────────────────────────────
1. 